# Heart Disease Prediction: Binary Classification

**Dataset:** UCI Cleveland Heart Disease — 303 patients, 13 clinical features (age, sex, chest pain type, resting BP, cholesterol, max heart rate, ST depression, etc.).

**Target:** Binary — presence (1) or absence (0) of heart disease.

**Objective:** Compare Logistic Regression and Random Forest classifiers with sklearn Pipelines, evaluate with confusion matrices and ROC curves, and tune the best model with GridSearchCV.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay

## 1. Dataset & Preprocessing

In [ ]:
# Column names from UCI description
col_names = [
    "age", "sex", "cp", "trestbps", "chol",
    "fbs", "restecg", "thalach", "exang",
    "oldpeak", "slope", "ca", "thal", "target"
]

# Load the processed Cleveland data (comma-separated, no header)
df = pd.read_csv("data/heart_disease.csv", header=None, names=col_names)

print("Shape:", df.shape)
df.head()

In [ ]:
# Basic info / summary
df.info()
df.describe().T

### 1.1 Data Cleaning

Replace `?` missing values in `ca` and `thal` with NaN, impute with mode. Convert multi-class target (0–4) to binary: 0 = no disease, 1 = disease present.

In [ ]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Columns with missing values
print(df.isna().sum())

# Convert ca & thal to numeric
df["ca"] = pd.to_numeric(df["ca"], errors="coerce")
df["thal"] = pd.to_numeric(df["thal"], errors="coerce")

print("\nMissing after numeric conversion:")
print(df.isna().sum())

# Impute missing values in ca & thal with most frequent (mode)
imputer = SimpleImputer(strategy="most_frequent")
df[["ca", "thal"]] = imputer.fit_transform(df[["ca", "thal"]])

print("\nMissing after imputation:")
print(df.isna().sum())

In [ ]:
# Convert target to binary: 0 = no disease, 1 = disease
df["target"] = (df["target"] > 0).astype(int)

df["target"].value_counts()

## 2. Exploratory Data Analysis

In [ ]:
# 1) Distribution of age
plt.figure(figsize=(8,4))
sns.histplot(df["age"], bins=20, kde=True)
plt.title("Age distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:
# 2) Cholesterol by target (boxplot)
plt.figure(figsize=(8,4))
sns.boxplot(data=df, x="target", y="chol")
plt.title("Serum Cholesterol vs Heart Disease")
plt.xlabel("Heart disease (1=yes, 0=no)")
plt.ylabel("Cholesterol")
plt.show()

In [ ]:
# 3) Count of heart disease by sex
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="sex", hue="target")
plt.title("Heart Disease by Sex")
plt.xlabel("Sex (1=male, 0=female)")
plt.ylabel("Count")
plt.legend(title="Heart disease")
plt.show()

In [ ]:
# 4) Chest pain type vs target
plt.figure(figsize=(8,4))
sns.countplot(data=df, x="cp", hue="target")
plt.title("Chest Pain Type vs Heart Disease")
plt.xlabel("Chest pain type (cp)")
plt.ylabel("Count")
plt.legend(title="Heart disease")
plt.show()

In [ ]:
# 5) Correlation heatmap for numeric features
plt.figure(figsize=(10,8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# 6) Thalach (max heart rate) distribution by target
plt.figure(figsize=(8,4))
sns.kdeplot(data=df, x="thalach", hue="target", fill=True, common_norm=False)
plt.title("Max Heart Rate vs Heart Disease")
plt.xlabel("Max heart rate (thalach)")
plt.show()

## 3. Modelling Pipeline

Sklearn `Pipeline` with `ColumnTransformer`: StandardScaler for numeric features, OneHotEncoder for categorical. Stratified 80/20 train/test split.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

### 3.1 Logistic Regression

In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

log_reg_pipeline.fit(X_train, y_train)

y_pred_lr = log_reg_pipeline.predict(X_test)
y_prob_lr = log_reg_pipeline.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification report:\n", classification_report(y_test, y_pred_lr))

In [ ]:
# Confusion matrix heatmap for LR
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(5,4))
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# ROC curve for LR
RocCurveDisplay.from_estimator(log_reg_pipeline, X_test, y_test)
plt.title("Logistic Regression - ROC Curve")
plt.show()

### 3.2 Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        random_state=42,
        n_estimators=100
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification report:\n", classification_report(y_test, y_pred_rf))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens")
plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# ROC curve for RF
RocCurveDisplay.from_estimator(rf_pipeline, X_test, y_test)
plt.title("Random Forest - ROC Curve")
plt.show()

### 3.3 Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 4],
    "model__min_samples_leaf": [1, 2]
}

rf_base = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

In [ ]:
# Evaluate best RF on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

print("=== Tuned Random Forest ===")
print("Test accuracy:", accuracy_score(y_test, y_pred_best))
print("Test ROC-AUC:", roc_auc_score(y_test, y_prob_best))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_best))
print("\nClassification report:\n", classification_report(y_test, y_pred_best))

## Conclusions

| Model | Notes |
|---|---|
| Logistic Regression | Strong baseline; interpretable coefficients |
| Random Forest | Better captures non-linear feature interactions |
| Tuned Random Forest | GridSearchCV improves generalisation |

**Key findings:**
- Max heart rate (`thalach`) and chest pain type (`cp`) are the strongest predictors of heart disease
- Sex and ST depression (`oldpeak`) also show clear separation between classes
- Both models benefit from the preprocessing pipeline — StandardScaler is critical for Logistic Regression convergence
- ROC curves confirm Random Forest outperforms Logistic Regression on AUC after tuning